<a href="https://colab.research.google.com/github/amosagekouassi-source/DI-Bootcamp/blob/main/DailyChallenge_J3_W7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Installation

In [ ]:
!pip install -q -U langchain
!pip install -q -U langchain-community
!pip install -q -U langchain-huggingface
!pip install -q -U transformers
!pip install -q -U sentence-transformers
!pip install -q -U datasets
!pip install -q -U faiss-cpu

##Load Dataset

In [ ]:
from langchain_community.document_loaders import HuggingFaceDatasetLoader

dataset_name = "databricks/databricks-dolly-15k"

# Le parametre correct est 'path' au lieu de 'dataset_name'
loader = HuggingFaceDatasetLoader(
    path=dataset_name,
    page_content_column="context"
)

documents = loader.load()

print(len(documents))

##Cut the document

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)

docs = splitter.split_documents(documents)

##Embedding

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

##Vectors store

In [ ]:
from langchain_community.vectorstores import FAISS

db = FAISS.from_documents(
    docs,
    embedding_model
)

retriever = db.as_retriever(
    search_kwargs={"k":4}
)

##Model loading

In [ ]:
from transformers import AutoTokenizer
from transformers import AutoModelForSeq2SeqLM
from transformers import pipeline

from langchain_huggingface import HuggingFacePipeline

model_name = "google/flan-T5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=128,
    do_sample=False,
    device=-1
)

llm = HuggingFacePipeline(
    pipeline=pipe
)

##Build RAG

In [ ]:
from langchain_classic.chains import RetrievalQA

qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)

##Test

In [ ]:
question = "What is cheesemaking?"

response = qa.invoke(
    {"query":question}
)

print(response["result"])